## Importar librerías

In [ ]:
from pathlib import Path
import pandas as pd

## Mejorar visualización de los dataframes

In [ ]:
# Que muestre todas las columnas
pd.options.display.max_columns = None

# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

## Establecer las rutas

In [ ]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "dane" / "exports"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "dane" / "exports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Importar y combinar los archivos mensuales

In [ ]:
archivos_csv = sorted(RAW_DIR.rglob("*.csv"))

print("Archivos encontrados:", len(archivos_csv))

In [ ]:
columnas_dane = [
    "FECH",
    "COD_PAI4",
    "POSAR",
    "PNK",
    "FOBDOL"
]

bases = []

for archivo in archivos_csv:
    temp = pd.read_csv(
        archivo,
        sep=";",
        dtype=str,
        usecols=lambda x: x.strip() in columnas_dane
    )

    temp.columns = temp.columns.str.strip()
    bases.append(temp)

df = pd.concat(bases, axis=0, ignore_index=True)

df.shape

In [ ]:
df.head()

## Preparar y limpiar las variables

In [ ]:
# Convertir las variables numéricas
for columna in ["PNK", "FOBDOL"]:
    df[columna] = pd.to_numeric(
        df[columna]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

In [ ]:
# Crear las variables de tiempo
df["year"] = 2000 + df["FECH"].str[:2].astype(int)
df["month"] = df["FECH"].str[2:].astype(int)

df["date"] = pd.to_datetime(
    dict(year=df["year"], month=df["month"], day=1)
)

In [ ]:
# Crear los códigos de producto
df["hs6"] = df["POSAR"].str.zfill(10).str[:6]
df["hs4"] = df["hs6"].str[:4]
df["hs2"] = df["hs6"].str[:2]

# Homologar las demás variables
df["flow"] = "X"
df["hs_revision"] = "HS2022"
df["partner_iso3"] = df["COD_PAI4"].str.strip().str.upper()
df["trade_value_usd"] = df["FOBDOL"]
df["net_weight_kg"] = df["PNK"]
df["source"] = "DANE"

## Agregar a la unidad de análisis final

In [ ]:
claves = [
    "year",
    "month",
    "date",
    "flow",
    "hs6",
    "hs4",
    "hs2",
    "hs_revision",
    "partner_iso3",
    "source"
]

dane = (
    df
    .groupby(claves, as_index=False, dropna=False)
    .agg(
        trade_value_usd=("trade_value_usd", lambda x: x.sum(min_count=1)),
        net_weight_kg=("net_weight_kg", lambda x: x.sum(min_count=1))
    )
    .sort_values(["date", "partner_iso3", "hs6"])
    .reset_index(drop=True)
)

dane.shape

## Validar y exportar

In [ ]:
print("Primer periodo:", dane["date"].min())
print("Último periodo:", dane["date"].max())
print("Meses:", dane["date"].nunique())
print(
    "Duplicados:",
    dane.duplicated(
        subset=["date", "flow", "hs6", "partner_iso3"]
    ).sum()
)

In [ ]:
archivo_dane = PROCESSED_DIR / "dane_exports_2025_2026.parquet"

dane.to_parquet(
    archivo_dane,
    index=False
)

archivo_dane

## Importar librerías

In [ ]:
from pathlib import Path
import os
import pandas as pd
import comtradeapicall
from dotenv import load_dotenv

## Mejorar visualización de los dataframes

In [ ]:
pd.options.display.max_columns = None
pd.options.display.float_format = '{:,.2f}'.format

## Establecer las rutas y la API key

In [ ]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "comtrade" / "exports"
RAW_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")

API_KEY = os.getenv("COMTRADE_API_KEY")

if not API_KEY:
    raise ValueError("No se encontró COMTRADE_API_KEY en el archivo .env")

## Definir los periodos

In [ ]:
periodos = [
    f"{year}{month:02d}"
    for year in range(2017, 2025)
    for month in range(1, 13)
]

print("Meses por descargar:", len(periodos))

## Descargar los datos mensuales

In [ ]:
def descargar_mes(periodo):
    return comtradeapicall.getFinalData(
        API_KEY,
        typeCode="C",
        freqCode="M",
        clCode="HS",
        period=periodo,
        reporterCode="170",
        cmdCode="AG6",
        flowCode="X",
        partnerCode=None,
        partner2Code=None,
        customsCode=None,
        motCode=None,
        maxRecords=250000,
        format_output="JSON",
        aggregateBy=None,
        breakdownMode="classic",
        countOnly=None,
        includeDesc=True
    )

In [ ]:
for periodo in periodos:

    archivo = (
        RAW_DIR
        / f"comtrade_exports_COL_{periodo}_HS6.parquet"
    )

    # No volver a descargar archivos que ya existen
    if archivo.exists():
        continue

    print("Descargando:", periodo)

    df_mes = descargar_mes(periodo)

    if df_mes.empty:
        print("Sin datos:", periodo)
        continue

    df_mes.to_parquet(
        archivo,
        index=False
    )

## Validar la descarga

In [ ]:
archivos = sorted(
    RAW_DIR.glob("comtrade_exports_COL_*_HS6.parquet")
)

print("Archivos disponibles:", len(archivos))
print("Primer archivo:", archivos[0].name)
print("Último archivo:", archivos[-1].name)

## Importar librerías

In [ ]:
from pathlib import Path
import pandas as pd

## Mejorar visualización de los dataframes

In [ ]:
pd.options.display.max_columns = None
pd.options.display.float_format = '{:,.2f}'.format

## Establecer las rutas

In [ ]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "comtrade" / "exports"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "comtrade" / "exports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Importar y combinar los archivos mensuales

In [ ]:
archivos = sorted(
    archivo
    for archivo in RAW_DIR.glob(
        "comtrade_exports_COL_*_HS6.parquet"
    )
    if 2017 <= int(
        archivo.stem.split("_")[3][:4]
    ) <= 2024
)

print("Archivos encontrados:", len(archivos))
print("Primer archivo:", archivos[0].name)
print("Último archivo:", archivos[-1].name)

In [ ]:
columnas = [
    "refYear",
    "refMonth",
    "flowCode",
    "partnerCode",
    "partnerISO",
    "cmdCode",
    "cmdDesc",
    "aggrLevel",
    "primaryValue",
    "netWgt",
    "classificationCode"
]

In [ ]:
bases = []
omitidos = []

for archivo in archivos:

    try:
        temp = pd.read_parquet(
            archivo,
            columns=columnas
        )

        if not temp.empty:
            bases.append(temp)

    except Exception:
        omitidos.append(archivo.name)

df = pd.concat(
    bases,
    axis=0,
    ignore_index=True
)

print("Archivos importados:", len(bases))
print("Archivos omitidos:", len(omitidos))
print("Dimensión:", df.shape)

## Preparar y limpiar las variables

In [ ]:
# Conservar únicamente productos HS6 y excluir el agregado World
df = df.loc[
    (df["aggrLevel"] == 6) &
    (df["partnerCode"] != 0)
].copy()

In [ ]:
# Crear las variables de tiempo
df["year"] = df["refYear"].astype(int)
df["month"] = df["refMonth"].astype(int)

df["date"] = pd.to_datetime(
    dict(year=df["year"], month=df["month"], day=1)
)

In [ ]:
# Crear las variables de producto
df["hs6"] = df["cmdCode"].astype(str).str.zfill(6)
df["hs6_name"] = df["cmdDesc"].astype("string").str.strip()
df["hs4"] = df["hs6"].str[:4]
df["hs2"] = df["hs6"].str[:2]

# Homologar las demás variables
df["flow"] = df["flowCode"]
df["partner_iso3"] = df["partnerISO"].astype("string").str.strip().str.upper()
df["trade_value_usd"] = pd.to_numeric(df["primaryValue"], errors="coerce")
df["net_weight_kg"] = pd.to_numeric(df["netWgt"], errors="coerce")
df["source"] = "UN Comtrade"

df["hs_revision"] = df["classificationCode"].map({
    "H5": "HS2017",
    "H6": "HS2022"
})

## Agregar a la unidad de análisis final

In [ ]:
claves = [
    "year",
    "month",
    "date",
    "flow",
    "hs6",
    "hs6_name",
    "hs4",
    "hs2",
    "hs_revision",
    "partner_iso3",
    "source"
]

comtrade = (
    df
    .groupby(claves, as_index=False, dropna=False)
    .agg(
        trade_value_usd=("trade_value_usd", lambda x: x.sum(min_count=1)),
        net_weight_kg=("net_weight_kg", lambda x: x.sum(min_count=1))
    )
    .sort_values(["date", "partner_iso3", "hs6"])
    .reset_index(drop=True)
)

comtrade.shape

## Validar y exportar

In [ ]:
print("Primer periodo:", comtrade["date"].min())
print("Último periodo:", comtrade["date"].max())
print("Meses:", comtrade["date"].nunique())
print(
    "Duplicados:",
    comtrade.duplicated(
        subset=["date", "flow", "hs6", "partner_iso3"]
    ).sum()
)

In [ ]:
archivo_comtrade = (
    PROCESSED_DIR
    / "comtrade_exports_COL_2017_2024_HS6.parquet"
)

comtrade.to_parquet(
    archivo_comtrade,
    index=False
)

archivo_comtrade

## Importar librerías

In [ ]:
from pathlib import Path
import pandas as pd
import requests

## Mejorar visualización de los dataframes

In [ ]:
pd.options.display.max_columns = None
pd.options.display.float_format = '{:,.2f}'.format

## Establecer las rutas

In [ ]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DANE_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "dane"
    / "exports"
    / "dane_exports_2025_2026.parquet"
)

COMTRADE_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "comtrade"
    / "exports"
    / "comtrade_exports_COL_2017_2024_HS6.parquet"
)

REFERENCE_DIR = PROJECT_ROOT / "data" / "raw" / "dane" / "reference"
FINAL_DIR = PROJECT_ROOT / "data" / "processed" / "final"

REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

## Cargar las bases procesadas

In [ ]:
dane = pd.read_parquet(DANE_FILE)
comtrade = pd.read_parquet(COMTRADE_FILE)

dane.shape, comtrade.shape

## Crear un catálogo único de países

In [ ]:
archivo_paises = REFERENCE_DIR / "correlativa_aladi_m49.xlsx"

url_paises = (
    "https://microdatos.dane.gov.co/"
    "index.php/catalog/859/download/24685"
)

# Descargar la correlativa oficial solo si todavía no existe
if not archivo_paises.exists():
    respuesta = requests.get(url_paises, timeout=60)
    respuesta.raise_for_status()
    archivo_paises.write_bytes(respuesta.content)

In [ ]:
paises = pd.read_excel(
    archivo_paises,
    sheet_name="ALADI_M49",
    header=4,
    dtype=str
)

paises = paises.iloc[:, :5]

paises.columns = [
    "codigo_aladi",
    "pais_aladi",
    "codigo_m49",
    "partner_iso3",
    "partner_name"
]

paises["partner_iso3"] = (
    paises["partner_iso3"]
    .astype("string")
    .str.strip()
    .str.upper()
)

paises["partner_name"] = (
    paises["partner_name"]
    .astype("string")
    .str.strip()
)

paises = (
    paises.loc[
        paises["partner_iso3"].str.fullmatch(
            r"[A-Z]{3}",
            na=False
        ),
        ["partner_iso3", "partner_name"]
    ]
    .drop_duplicates("partner_iso3")
)

iso3_validos = set(paises["partner_iso3"])

nombres_paises = dict(
    zip(
        paises["partner_iso3"],
        paises["partner_name"]
    )
)

In [ ]:
# Aplicar la misma clasificación de destinos a las dos fuentes
for base in [comtrade, dane]:

    iso = (
        base["partner_iso3"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    base["partner_iso3"] = iso.where(
        iso.isin(iso3_validos) & iso.ne("XCF"),
        "OTHER_SPECIAL"
    )

    base["partner_name"] = (
        base["partner_iso3"]
        .map(nombres_paises)
        .fillna("Otros destinos especiales")
    )

## Agregar la descripción HS6 a DANE

In [ ]:
url_hs2022 = (
    "https://comtradeapi.un.org/"
    "files/v1/app/reference/H6.json"
)

respuesta = requests.get(url_hs2022, timeout=60)
respuesta.raise_for_status()

catalogo_hs6 = pd.DataFrame(
    respuesta.json()["results"]
)

catalogo_hs6 = catalogo_hs6.loc[
    catalogo_hs6["aggrlevel"] == 6,
    ["id", "text"]
].copy()

catalogo_hs6["hs6"] = (
    catalogo_hs6["id"]
    .astype("string")
    .str.zfill(6)
)

catalogo_hs6["hs6_name"] = (
    catalogo_hs6["text"]
    .astype("string")
    .str.replace(
        r"^\d{6}\s*-\s*",
        "",
        regex=True
    )
    .str.strip()
)

catalogo_hs6 = (
    catalogo_hs6[
        ["hs6", "hs6_name"]
    ]
    .drop_duplicates("hs6")
)

In [ ]:
dane = dane.merge(
    catalogo_hs6,
    on="hs6",
    how="left"
)

# Códigos especiales utilizados en la nomenclatura colombiana
nombres_hs_colombia = {
    "980500": "Household effects",
    "980600": "Works of art under headings 97.01, 97.02 and 97.03, imported by the author",
    "980710": "Postal traffic",
    "980720": "Express delivery or courier shipments"
}

dane["hs6_name"] = (
    dane["hs6_name"]
    .fillna(
        dane["hs6"].map(nombres_hs_colombia)
    )
)

print(
    "HS6 sin descripción:",
    dane["hs6_name"].isna().sum()
)

## Combinar las dos fuentes

In [ ]:
columnas_finales = [
    "year",
    "month",
    "date",
    "flow",
    "hs6",
    "hs6_name",
    "hs4",
    "hs2",
    "hs_revision",
    "partner_iso3",
    "partner_name",
    "trade_value_usd",
    "net_weight_kg",
    "source"
]

base_final = pd.concat(
    [
        comtrade[columnas_finales],
        dane[columnas_finales]
    ],
    axis=0,
    ignore_index=True
)

base_final.shape

## Consolidar la categoría OTHER_SPECIAL

In [ ]:
claves = [
    "year",
    "month",
    "date",
    "flow",
    "hs6",
    "hs6_name",
    "hs4",
    "hs2",
    "hs_revision",
    "partner_iso3",
    "partner_name",
    "source"
]

base_final = (
    base_final
    .groupby(claves, as_index=False, dropna=False)
    .agg(
        trade_value_usd=("trade_value_usd", lambda x: x.sum(min_count=1)),
        net_weight_kg=("net_weight_kg", lambda x: x.sum(min_count=1))
    )
    .sort_values(["date", "partner_iso3", "hs6"])
    .reset_index(drop=True)
)

base_final.shape

## Validar la base final

In [ ]:
print("Filas:", len(base_final))
print("Columnas:", len(base_final.columns))
print("Meses:", base_final["date"].nunique())
print("Desde:", base_final["date"].min())
print("Hasta:", base_final["date"].max())

print(
    "Duplicados:",
    base_final.duplicated(
        subset=["date", "flow", "hs6", "partner_iso3"]
    ).sum()
)

print(
    "HS6 sin nombre:",
    base_final["hs6_name"].isna().sum()
)

print(
    "Destinos geográficos:",
    base_final.loc[
        base_final["partner_iso3"] != "OTHER_SPECIAL",
        "partner_iso3"
    ].nunique()
)

In [ ]:
base_final.groupby("source").agg(
    fecha_inicial=("date", "min"),
    fecha_final=("date", "max"),
    meses=("date", "nunique"),
    registros=("date", "size")
)

## Exportar la base final

In [ ]:
FINAL_FILE = (
    FINAL_DIR
    / "exports_COL_2017_2026_HS6_country.parquet"
)

base_final.to_parquet(
    FINAL_FILE,
    index=False
)

FINAL_FILE

In [ ]:
final_check = pd.read_parquet(FINAL_FILE)

print("Dimensión final:", final_check.shape)
print(
    "Duplicados:",
    final_check.duplicated(
        subset=["date", "flow", "hs6", "partner_iso3"]
    ).sum()
)

final_check.head()

## Pipeline: construcción del panel Colombia-gravedad (2017-2024)
Fuentes: BACI (comercio bilateral, CEPII), Gravity Dataset (CEPII), PIB (World Bank API)
Salida: panel_colombia_gravedad.csv / .parquet — un país-socio por año, listo para EDA y modelado

## 1. Librerías

In [ ]:
import duckdb
import pandas as pd
import wbgapi as wb

## 2. Cargar BACI filtrado a Colombia

In [ ]:
# Usamos DuckDB para leer cada CSV desde disco y filtrar antes de traer nada a memoria:
# BACI completo son ~200 países x 5000 productos, no cabe en RAM cargado entero.
con = duckdb.connect()
COL = 170  # código BACI de Colombia (confirmado contra country_codes_V202601.csv)

años = range(2017, 2025)
dfs = []
for y in años:
    q = f"""
        SELECT * FROM read_csv_auto('BACI_HS17_V202601/BACI_HS17_Y{y}_*.csv')
        WHERE i = {COL} OR j = {COL}
    """
    dfs.append(con.execute(q).df())

baci_col = pd.concat(dfs, ignore_index=True)
baci_col["k"] = baci_col["k"].astype(str).str.zfill(6)  # evita perder ceros a la izquierda del código HS

## 3. Separar fuels (capítulo HS 27) de no-fuels, y exportaciones de importaciones

In [ ]:
baci_col["chapter"] = baci_col["k"].str[:2]
baci_col["fuel"] = baci_col["chapter"] == "27"

x_col = baci_col[baci_col["i"] == COL].copy()
m_col = baci_col[baci_col["j"] == COL].copy()


def agregar(df, partner_col, prefix):
    """Colapsa de nivel producto a nivel año-socio, separando total / fuels / no-fuels."""
    total = df.groupby(["t", partner_col])["v"].sum().rename(f"{prefix}_Total")
    fuels = df[df["fuel"]].groupby(["t", partner_col])["v"].sum().rename(f"{prefix}_Fuels")
    out = pd.concat([total, fuels], axis=1).fillna(0)
    out[f"{prefix}_NoComb"] = out[f"{prefix}_Total"] - out[f"{prefix}_Fuels"]
    return out.reset_index().rename(columns={partner_col: "partner"})


exports = agregar(x_col, "j", "X")
imports = agregar(m_col, "i", "M")

exports.to_parquet("exports_col.parquet")
imports.to_parquet("imports_col.parquet")

## 4. Cargar el Gravity Dataset (CEPII) — solo cubre hasta 2021

In [ ]:
grav = pd.read_csv("Gravity_csv_V202211/Gravity_V202211.csv")
grav_col = grav[(grav["iso3_o"] == "COL") & (grav["year"] >= 2017)].copy()

# El dataset trae filas duplicadas por año-país para algunos socios (ej. DEU, PAK, VNM)
# por códigos de país heredados de fusiones/divisiones históricas. Nos quedamos con el
# primer valor no nulo de cada grupo año-socio para colapsar a una sola fila limpia.
grav_col = grav_col.groupby(["year", "iso3_d"], as_index=False).first()

## 5. Extender variables fijas a 2022-2024

In [ ]:
# dist, contig, comlang_off, col_dep_ever no cambian en el tiempo: se repite el último
# valor disponible (2021). El PIB sí se actualiza con datos reales en el paso 6.
ultimo_año = grav_col[grav_col["year"] == 2021].copy()
extension = []
for y in [2022, 2023, 2024]:
    temp = ultimo_año.copy()
    temp["year"] = y
    extension.append(temp)

grav_extendido = pd.concat([grav_col] + extension, ignore_index=True)

## 6. PIB del socio desde la API del Banco Mundial

In [ ]:
# Necesario porque el Gravity Dataset no cubre 2022-2024; trae el valor real en vez de
# arrastrar el de 2021.
gdp = wb.data.DataFrame("NY.GDP.MKTP.KD", time=range(2017, 2025), labels=True).reset_index()
gdp_long = gdp.melt(id_vars=["economy", "Country"], var_name="year", value_name="gdp")
gdp_long["year"] = gdp_long["year"].str.replace("YR", "").astype(int)
gdp_long = gdp_long.rename(columns={"economy": "iso3"})

grav_extendido.to_parquet("grav_extendido.parquet")
gdp_long.to_parquet("gdp_long.parquet")

## 7. Traducir el código de socio de BACI (numérico) a ISO3 (formato del Gravity Dataset)

In [ ]:
country_codes = pd.read_csv("BACI_HS17_V202601/country_codes_V202601.csv")
code_to_iso3 = dict(zip(country_codes["country_code"], country_codes["country_iso3"]))

exports["iso3_d"] = exports["partner"].map(code_to_iso3)
imports["iso3_d"] = imports["partner"].map(code_to_iso3)

comercio = exports.merge(imports, on=["t", "partner", "iso3_d"], how="outer")
comercio = comercio.rename(columns={"t": "year"})

## 8. Unir gravedad + comercio + PIB actualizado

In [ ]:
panel = grav_extendido.merge(comercio, on=["year", "iso3_d"], how="left")

gdp_partner = gdp_long.rename(columns={"iso3": "iso3_d", "gdp": "gdp_d_actualizado"})
panel = panel.merge(gdp_partner[["iso3_d", "year", "gdp_d_actualizado"]], on=["iso3_d", "year"], how="left")
panel["gdp_d"] = panel["gdp_d_actualizado"].combine_first(panel["gdp_d"])
panel = panel.drop(columns=["gdp_d_actualizado"])

cols_comercio = ["X_Total", "X_Fuels", "X_NoComb", "M_Total", "M_Fuels", "M_NoComb"]
panel[cols_comercio] = panel[cols_comercio].fillna(0)  # sin comercio registrado = 0, no dato faltante

## 9. Descartar columnas de metadata del Gravity Dataset que no se usan en el modelo

In [ ]:
cols_a_botar = [
    "distw_harmonic_jh", "distw_arithmetic_jh",
    "gdp_source_o", "gdp_source_d", "pop_source_o", "pop_source_d",
    "empire", "col_dep_end_year", "col_dep_end_conflict", "pop_pwt_d",
]
panel = panel.drop(columns=[c for c in cols_a_botar if c in panel.columns])

## 10. Variables de interés del modelo

In [ ]:
# Nombres confirmados contra el codebook del Gravity Dataset:
#   - col_dep_ever: relación colonial histórica (verificado: España-Colombia = 1)
#   - fta_wto: TLC vigente notificado a la OMC
COLONIA_col = "col_dep_ever"
FTA_col = "fta_wto"

vars_modelo = [
    "X_Total", "M_Total", "X_Fuels", "M_Fuels", "X_NoComb", "M_NoComb",
    "dist", "contig", "comlang_off", COLONIA_col,
    "gdp_d", FTA_col,
]

## 11. Eliminar entidades que ya no existen y microestados/territorios sin peso comercial propio

In [ ]:
eliminar = [
    "ANT", "CSK", "DDR", "SCG", "SUN", "VDR", "YMD", "YUG",            # entidades disueltas antes de 2017
    "AIA", "BES", "CCK", "COK", "CXR", "ERI", "ESH", "FLK", "GIB",
    "GLP", "GUF", "IOT", "MSR", "MTQ", "MYT", "NFK", "NIU", "PCN",
    "REU", "SHN", "SPM", "TKL", "VAT", "VGB", "WLF", "LIE", "NCL",
    "ASM", "GUM", "MNP", "SMR",                                        # microestados/territorios
    "PRK", "SYR", "SSD", "GRL",                                        # aislamiento/conflicto, comercio marginal
    "CUW", "SXM", "MCO",                                                # territorios pequeños adicionales
]
# TWN queda fuera de esta lista a propósito: es un socio comercial real, solo le falta
# el PIB del Banco Mundial por temas de reconocimiento político. Decisión pendiente del
# grupo: completarlo a mano (FMI) o excluirlo también.

panel_limpio = panel[~panel["iso3_d"].isin(eliminar)].copy()

print("Filas:", panel_limpio.shape[0], "| Países:", panel_limpio["iso3_d"].nunique())
print(panel_limpio[vars_modelo].isna().sum())

## 12. Exportar panel final

In [ ]:
panel_limpio.to_csv("prueba.csv", index=False)
panel_limpio.to_parquet("prueba.parquet")

## Importar librerías

In [ ]:
from pathlib import Path
import os
import time

import pandas as pd
import comtradeapicall
from dotenv import load_dotenv

## Mejorar visualización de los dataframes

In [ ]:
# Que muestre todas las columnas
pd.options.display.max_columns = None

# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

## Establecer la ruta de los datasets

In [ ]:
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data").is_dir()
)

INTERIM_DIR = PROJECT_ROOT / "data" / "interim" / "final"
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "comtrade" / "imports_demand"
FINAL_DIR = PROJECT_ROOT / "data" / "processed" / "final"

DATOS_FILE = INTERIM_DIR / "datoslimpios.csv"
OUTPUT_FILE = FINAL_DIR / "variables_model_2017_2024.parquet"

RAW_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

## Importar la base de variables

In [ ]:
datos = pd.read_csv(DATOS_FILE)

datos.head()

In [ ]:
print("Dimensión:", datos.shape)
print("Periodo:", datos["year"].min(), "→", datos["year"].max())
print("Países:", datos["iso3_d"].nunique())

print(
    "Duplicados año-país:",
    datos.duplicated(
        subset=["year", "iso3_d"]
    ).sum()
)

## Configurar UN Comtrade

In [ ]:
load_dotenv(PROJECT_ROOT / ".env")

API_KEY = os.getenv("COMTRADE_API_KEY")

if not API_KEY:
    raise ValueError("No se encontró COMTRADE_API_KEY en el archivo .env")

In [ ]:
paises = (
    datos["iso3_d"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("Países por consultar:", len(paises))

## Descargar la demanda importadora

In [ ]:
tamano_lote = 20
errores = []

for year in range(2017, 2025):

    for inicio in range(0, len(paises), tamano_lote):

        lote = paises[inicio:inicio + tamano_lote]
        numero_lote = inicio // tamano_lote + 1

        archivo = (
            RAW_DIR
            / f"imports_demand_{year}_batch_{numero_lote:02d}.parquet"
        )

        # Saltar lotes ya descargados
        if archivo.exists():
            continue

        reporter_codes = (
            comtradeapicall
            .convertCountryIso3ToCode(
                ",".join(lote)
            )
        )

        print(year, numero_lote, len(lote))

        temp = None

        # Intentar máximo tres veces
        for intento in range(3):

            try:
                temp = comtradeapicall.getFinalData(
                    API_KEY,
                    typeCode="C",
                    freqCode="A",
                    clCode="HS",
                    period=str(year),
                    reporterCode=reporter_codes,
                    cmdCode="AG6",
                    flowCode="M",
                    partnerCode="0",
                    partner2Code=None,
                    customsCode=None,
                    motCode=None,
                    maxRecords=250000,
                    format_output="JSON",
                    aggregateBy=None,
                    breakdownMode="classic",
                    countOnly=None,
                    includeDesc=True
                )

                if isinstance(temp, pd.DataFrame):
                    break

            except Exception as error:
                print(f"Intento {intento + 1} falló:", error)

            time.sleep(5)

        if isinstance(temp, pd.DataFrame) and not temp.empty:

            temp.to_parquet(
                archivo,
                index=False
            )

        else:

            errores.append({
                "year": year,
                "batch": numero_lote
            })

            print("Lote no descargado:", year, numero_lote)

        time.sleep(2)

In [ ]:
archivos_demanda = sorted(
    RAW_DIR.glob("imports_demand_*.parquet")
)

print("Lotes con error:", len(errores))
print("Archivos descargados:", len(archivos_demanda))

## Combinar los archivos descargados

In [ ]:
columnas_demanda = [
    "refYear",
    "reporterISO",
    "cmdCode",
    "cmdDesc",
    "aggrLevel",
    "partnerCode",
    "primaryValue"
]

bases = []

for archivo in archivos_demanda:

    temp = pd.read_parquet(
        archivo,
        columns=columnas_demanda
    )

    bases.append(temp)

demanda = pd.concat(
    bases,
    ignore_index=True
)

demanda.shape

## Preparar la demanda importadora

In [ ]:
demanda = demanda.loc[
    (demanda["aggrLevel"] == 6) &
    (demanda["partnerCode"] == 0)
].copy()

demanda = demanda[
    [
        "refYear",
        "reporterISO",
        "cmdCode",
        "cmdDesc",
        "primaryValue"
    ]
].copy()

demanda.columns = [
    "year",
    "iso3_d",
    "hs6",
    "hs6_name",
    "import_demand_usd"
]

In [ ]:
demanda["year"] = demanda["year"].astype(int)

demanda["hs6"] = (
    demanda["hs6"]
    .astype(str)
    .str.zfill(6)
)

demanda["import_demand_usd"] = pd.to_numeric(
    demanda["import_demand_usd"],
    errors="coerce"
)

demanda["hs_revision"] = demanda["year"].apply(
    lambda x: "HS2017" if x <= 2021 else "HS2022"
)

demanda.head()

In [ ]:
print("Dimensión:", demanda.shape)
print("Periodo:", demanda["year"].min(), "→", demanda["year"].max())
print("Países:", demanda["iso3_d"].nunique())
print("HS6:", demanda["hs6"].nunique())

print(
    "Duplicados año-país-HS6:",
    demanda.duplicated(
        subset=["year", "iso3_d", "hs6"]
    ).sum()
)

## Revisar cobertura de países

In [ ]:
resumen_anual = (
    demanda
    .groupby("year")
    .agg(
        registros=("hs6", "size"),
        paises=("iso3_d", "nunique"),
        productos=("hs6", "nunique"),
        demanda_usd=("import_demand_usd", "sum")
    )
)

resumen_anual

In [ ]:
paises_sin_demanda = sorted(
    set(datos["iso3_d"].dropna())
    -
    set(demanda["iso3_d"].dropna())
)

print("Países sin información en Comtrade:", len(paises_sin_demanda))
paises_sin_demanda

## Combinar con las variables del modelo

In [ ]:
demanda_modelo = demanda[
    [
        "year",
        "iso3_d",
        "hs6",
        "hs6_name",
        "hs_revision",
        "import_demand_usd"
    ]
].copy()

variables_model = demanda_modelo.merge(
    datos,
    on=["year", "iso3_d"],
    how="inner",
    validate="many_to_one"
)

variables_model.shape

In [ ]:
columnas_inicio = [
    "year",
    "iso3_d",
    "hs6",
    "hs6_name",
    "hs_revision",
    "import_demand_usd"
]

columnas_resto = [
    columna
    for columna in variables_model.columns
    if columna not in columnas_inicio
]

variables_model = variables_model[
    columnas_inicio + columnas_resto
]

variables_model.head()

## Validar la base final

In [ ]:
print("Dimensión:", variables_model.shape)
print("Periodo:", variables_model["year"].min(), "→", variables_model["year"].max())
print("Países:", variables_model["iso3_d"].nunique())
print("HS6:", variables_model["hs6"].nunique())

print(
    "Duplicados:",
    variables_model.duplicated(
        subset=["year", "iso3_d", "hs6"]
    ).sum()
)

print(
    "Demanda faltante:",
    variables_model["import_demand_usd"].isna().sum()
)

print(
    "Demanda negativa:",
    (variables_model["import_demand_usd"] < 0).sum()
)

## Exportar la base final

In [ ]:
variables_model.to_parquet(
    OUTPUT_FILE,
    index=False
)

print("Archivo guardado:")
print(OUTPUT_FILE)

print(
    "Dimensión final:",
    variables_model.shape
)

In [ ]:
variables_check = pd.read_parquet(OUTPUT_FILE)

print("Dimensión:", variables_check.shape)

print(
    "Duplicados:",
    variables_check.duplicated(
        subset=["year", "iso3_d", "hs6"]
    ).sum()
)

print(
    "Demanda faltante:",
    variables_check["import_demand_usd"].isna().sum()
)

## Importar librerías

In [ ]:
from pathlib import Path
import pandas as pd

## Mejorar visualización de los dataframes

In [ ]:
# Que muestre todas las columnas
pd.options.display.max_columns = None

# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

## Establecer las rutas

In [ ]:
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data").is_dir()
)

FINAL_DIR = PROJECT_ROOT / "data" / "processed" / "final"

EXPORTS_FILE = (
    FINAL_DIR
    / "exports_COL_2017_2026_HS6_country.parquet"
)

VARIABLES_FILE = (
    FINAL_DIR
    / "variables_model_2017_2024.parquet"
)

OUTPUT_FILE = (
    FINAL_DIR
    / "base_model_2017_2024.parquet"
)

## Importar las bases

In [ ]:
exports = pd.read_parquet(
    EXPORTS_FILE,
    columns=[
        "year",
        "hs6",
        "partner_iso3",
        "trade_value_usd",
        "net_weight_kg"
    ]
)

variables = pd.read_parquet(
    VARIABLES_FILE
)

print("Exportaciones:", exports.shape)
print("Variables:", variables.shape)

## Preparar las exportaciones

In [ ]:
exports = exports.loc[
    exports["year"].between(2017, 2024)
    &
    exports["partner_iso3"].ne("OTHER_SPECIAL")
].copy()

exports.shape

In [ ]:
exports_anual = (
    exports
    .groupby(
        [
            "year",
            "partner_iso3",
            "hs6"
        ],
        as_index=False
    )
    .agg(
        export_value_usd=(
            "trade_value_usd",
            "sum"
        ),
        export_weight_kg=(
            "net_weight_kg",
            lambda x: x.sum(min_count=1)
        )
    )
)

exports_anual.shape

In [ ]:
print(
    "Duplicados año-país-HS6:",
    exports_anual.duplicated(
        subset=[
            "year",
            "partner_iso3",
            "hs6"
        ]
    ).sum()
)

exports_anual.head()

## Combinar las bases

In [ ]:
base_model = variables.merge(
    exports_anual,
    left_on=[
        "year",
        "iso3_d",
        "hs6"
    ],
    right_on=[
        "year",
        "partner_iso3",
        "hs6"
    ],
    how="left",
    validate="one_to_one"
)

base_model.shape

## Preparar las variables de exportación

In [ ]:
base_model["export_value_usd"] = (
    base_model["export_value_usd"]
    .fillna(0)
)

base_model["export_weight_kg"] = (
    base_model["export_weight_kg"]
    .fillna(0)
)

base_model = base_model.drop(
    columns="partner_iso3"
)

In [ ]:
base_model["exported"] = (
    base_model["export_value_usd"] > 0
).astype(int)

base_model[
    [
        "year",
        "iso3_d",
        "hs6",
        "import_demand_usd",
        "export_value_usd",
        "export_weight_kg",
        "exported"
    ]
].head()

## Organizar las columnas principales

In [ ]:
columnas_inicio = [
    "year",
    "iso3_d",
    "hs6",
    "hs6_name",
    "hs_revision",
    "import_demand_usd",
    "export_value_usd",
    "export_weight_kg",
    "exported"
]

columnas_resto = [
    columna
    for columna in base_model.columns
    if columna not in columnas_inicio
]

base_model = base_model[
    columnas_inicio + columnas_resto
]

## Validar la base final

In [ ]:
print("Dimensión:", base_model.shape)
print("Periodo:", base_model["year"].min(), "→", base_model["year"].max())
print("Países:", base_model["iso3_d"].nunique())
print("HS6:", base_model["hs6"].nunique())

print(
    "Duplicados:",
    base_model.duplicated(
        subset=[
            "year",
            "iso3_d",
            "hs6"
        ]
    ).sum()
)

print(
    "Demanda faltante:",
    base_model["import_demand_usd"].isna().sum()
)

print(
    "Exportaciones faltantes:",
    base_model["export_value_usd"].isna().sum()
)

In [ ]:
resumen_exportaciones = pd.DataFrame({
    "registros": [
        len(base_model),
        (base_model["exported"] == 1).sum(),
        (base_model["exported"] == 0).sum()
    ]
},
index=[
    "Total",
    "Con exportaciones",
    "Sin exportaciones"
])

resumen_exportaciones

In [ ]:
resumen_anual = (
    base_model
    .groupby("year")
    .agg(
        registros=("hs6", "size"),
        paises=("iso3_d", "nunique"),
        productos=("hs6", "nunique"),
        demanda_usd=("import_demand_usd", "sum"),
        exportaciones_usd=("export_value_usd", "sum")
    )
)

resumen_anual

## Exportar la base final

In [ ]:
base_model.to_parquet(
    OUTPUT_FILE,
    index=False
)

print("Archivo guardado:")
print(OUTPUT_FILE)

print(
    "Dimensión final:",
    base_model.shape
)

In [ ]:
base_check = pd.read_parquet(
    OUTPUT_FILE
)

print("Dimensión:", base_check.shape)

print(
    "Duplicados:",
    base_check.duplicated(
        subset=[
            "year",
            "iso3_d",
            "hs6"
        ]
    ).sum()
)

print(
    "Demanda faltante:",
    base_check["import_demand_usd"].isna().sum()
)

print(
    "Exportaciones faltantes:",
    base_check["export_value_usd"].isna().sum()
)

base_check.head()

## Importar librerías

In [ ]:
import warnings
from pathlib import Path

import imageio.v2 as imageio
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from plotly.colors import sample_colorscale

warnings.filterwarnings("ignore")

## Mejorar visualización de los dataframes

In [ ]:
pd.options.display.max_columns = None
pd.options.display.float_format = "{:,.2f}".format

## Definir rutas del proyecto

In [ ]:
# Buscar automáticamente la raíz del proyecto desde VS Code/Jupyter
archivo_entrada = Path(
    "data/processed/final/exports_COL_2017_2026_HS6_country.parquet"
)

PROJECT_ROOT = next(
    (
        ruta
        for ruta in [Path.cwd(), *Path.cwd().parents]
        if (ruta / archivo_entrada).exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Abre VS Code en la carpeta proyecto_comercio_ia."
    )

DATA_FILE = PROJECT_ROOT / archivo_entrada
OUTPUT_DIR = PROJECT_ROOT / "notebook" / "graft"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Proyecto:", PROJECT_ROOT)
print("Entrada:", DATA_FILE)
print("Salidas:", OUTPUT_DIR)

## Cargar el dataset

In [ ]:
df = pd.read_parquet(DATA_FILE)

print("Dimensiones:", df.shape)
df.head()

## Preparar exportaciones por año y país destino

In [ ]:
exports_anuales = (
    df.groupby(
        ["year", "partner_iso3", "partner_name"],
        as_index=False
    )
    .agg(
        trade_value_usd=("trade_value_usd", "sum"),
        net_weight_kg=("net_weight_kg", "sum")
    )
    .sort_values(
        ["year", "trade_value_usd"],
        ascending=[True, False]
    )
)

destinos = exports_anuales[
    exports_anuales["partner_iso3"]
    .str.fullmatch(r"[A-Z]{3}", na=False)
].copy()

top_destinos = (
    destinos.groupby("year", group_keys=False)
    .head(15)
    .reset_index(drop=True)
)

cobertura = (
    df.groupby("year")["month"]
    .nunique()
    .rename("meses_disponibles")
)

years_interactive = sorted(top_destinos["year"].unique())
years_export = sorted(
    cobertura[cobertura.eq(12)].index.tolist()
)

print("Años interactivos:", years_interactive)
print("Años completos exportados:", years_export)

top_destinos.head(15)

## Coordenadas continentales

Para evitar puntos desplazados por territorios insulares, se usa el polígono territorial principal de cada país.  
Además, se conservan las correcciones manuales utilizadas para Estados Unidos, Chile y Santa Lucía.


In [ ]:
world = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/"
    "110m_cultural/ne_110m_admin_0_countries.zip"
)

mainland = (
    world[["ADM0_A3", "geometry"]]
    .explode(index_parts=False)
    .copy()
)

mainland["area"] = mainland.geometry.area

mainland = (
    mainland.sort_values("area", ascending=False)
    .drop_duplicates("ADM0_A3")
    .drop(columns="area")
)

points = mainland.geometry.representative_point()

coords = (
    mainland.assign(
        lon=points.x,
        lat=points.y
    )
    [["ADM0_A3", "lon", "lat"]]
    .rename(columns={"ADM0_A3": "partner_iso3"})
)

map_data = top_destinos.merge(
    coords,
    on="partner_iso3",
    how="left"
)

coordenadas_manual = {
    "USA": (39.80, -98.60),
    "CHL": (-33.45, -70.67),
    "LCA": (13.91, -60.98),
}

for iso3, (lat, lon) in coordenadas_manual.items():
    map_data.loc[
        map_data["partner_iso3"].eq(iso3),
        ["lat", "lon"]
    ] = [lat, lon]

faltantes = (
    map_data[
        map_data[["lat", "lon"]]
        .isna()
        .any(axis=1)
    ]
    [["partner_iso3", "partner_name"]]
    .drop_duplicates()
)

faltantes

## Crear el mapa interactivo

In [ ]:
def traces_year(year):

    m = map_data.query("year == @year").copy()

    x = np.log1p(m["trade_value_usd"])

    if x.max() == x.min():
        m["heat"] = 0.5
    else:
        m["heat"] = (
            (x - x.min())
            / (x.max() - x.min())
        )

    m["width"] = 1.2 + 2.8 * m["heat"]

    m["color"] = [
        sample_colorscale("Reds", [v])[0]
        for v in m["heat"]
    ]

    traces = [
        go.Choropleth(
            locations=m["partner_iso3"],
            locationmode="ISO-3",
            z=m["heat"],
            customdata=m[
                ["partner_name", "trade_value_usd"]
            ],
            colorscale="Reds",
            zmin=0,
            zmax=1,
            marker_line_width=0.4,
            colorbar=dict(
                title="Exportaciones"
            ),
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "US$ %{customdata[1]:,.0f}"
                "<extra></extra>"
            )
        )
    ]

    for _, r in m.iterrows():

        traces.append(
            go.Scattergeo(
                lon=[-74.3, r["lon"]],
                lat=[4.6, r["lat"]],
                mode="lines",
                line=dict(
                    width=r["width"],
                    color=r["color"]
                ),
                hoverinfo="skip",
                showlegend=False
            )
        )

    traces.append(
        go.Scattergeo(
            lon=m["lon"],
            lat=m["lat"],
            mode="markers",
            text=m["partner_name"],
            customdata=m["trade_value_usd"],
            marker=dict(
                size=7,
                color="black"
            ),
            hovertemplate=(
                "<b>%{text}</b><br>"
                "US$ %{customdata:,.0f}"
                "<extra></extra>"
            ),
            showlegend=False
        )
    )

    traces.append(
        go.Scattergeo(
            lon=[-74.3],
            lat=[4.6],
            mode="markers",
            text=["Colombia"],
            marker=dict(
                size=10,
                color="black"
            ),
            hovertemplate=(
                "<b>Colombia</b>"
                "<extra></extra>"
            ),
            showlegend=False
        )
    )

    return traces

In [ ]:
frames = [
    go.Frame(
        data=traces_year(year),
        name=str(year)
    )
    for year in years_interactive
]

fig = go.Figure(
    data=traces_year(years_interactive[0]),
    frames=frames
)

fig.update_layout(
    title=(
        "Principales destinos de las "
        "exportaciones de Colombia"
    ),
    geo=dict(
        projection_type="natural earth",
        showland=True,
        landcolor="rgb(240, 240, 240)",
        showcountries=True,
        countrycolor="white",
        showocean=True,
        oceancolor="white"
    ),
    height=650,
    margin=dict(
        l=0,
        r=0,
        t=60,
        b=0
    ),
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            x=0.05,
            y=0,
            buttons=[
                dict(
                    label="▶",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(
                                duration=900,
                                redraw=True
                            ),
                            transition=dict(
                                duration=300
                            ),
                            fromcurrent=True
                        )
                    ]
                ),
                dict(
                    label="❚❚",
                    method="animate",
                    args=[
                        [None],
                        dict(
                            frame=dict(
                                duration=0,
                                redraw=False
                            ),
                            mode="immediate"
                        )
                    ]
                )
            ]
        )
    ],
    sliders=[
        dict(
            active=0,
            currentvalue=dict(
                prefix="Año: "
            ),
            steps=[
                dict(
                    label=str(year),
                    method="animate",
                    args=[
                        [str(year)],
                        dict(
                            mode="immediate",
                            frame=dict(
                                duration=500,
                                redraw=True
                            ),
                            transition=dict(
                                duration=200
                            )
                        )
                    ]
                )
                for year in years_interactive
            ]
        )
    ]
)

fig.show()

## Exportar mapas anuales

Se exportan únicamente los años con 12 meses observados.  
Con la base actual, esto genera **9 imágenes: 2017–2025**.  
El año 2026 permanece disponible en el gráfico interactivo, pero no se mezcla con los años completos en el video.


In [ ]:
cmap = plt.cm.Reds

for year in years_export:

    m = map_data.query(
        "year == @year"
    ).copy()

    x = np.log1p(
        m["trade_value_usd"]
    )

    if x.max() == x.min():
        m["heat"] = 0.5
    else:
        m["heat"] = (
            (x - x.min())
            / (x.max() - x.min())
        )

    m["width"] = (
        1.2 + 2.8 * m["heat"]
    )

    fig, ax = plt.subplots(
        figsize=(12.8, 7.2),
        dpi=100
    )

    world.plot(
        ax=ax,
        color="#f0f0f0",
        edgecolor="white",
        linewidth=0.5
    )

    countries = world.merge(
        m[
            ["partner_iso3", "heat"]
        ],
        left_on="ADM0_A3",
        right_on="partner_iso3"
    )

    countries.plot(
        ax=ax,
        color=[
            cmap(v)
            for v in countries["heat"]
        ],
        edgecolor="white",
        linewidth=0.5
    )

    for _, r in m.iterrows():

        ax.plot(
            [-74.3, r["lon"]],
            [4.6, r["lat"]],
            color=cmap(r["heat"]),
            linewidth=r["width"],
            zorder=3
        )

    ax.scatter(
        m["lon"],
        m["lat"],
        color="black",
        s=28,
        zorder=4
    )

    ax.scatter(
        -74.3,
        4.6,
        color="black",
        s=55,
        zorder=5
    )

    ax.set_xlim(-180, 180)
    ax.set_ylim(-60, 90)
    ax.axis("off")

    ax.set_title(
        "Principales destinos de las "
        f"exportaciones de Colombia — {year}",
        fontsize=16,
        pad=15
    )

    fig.subplots_adjust(
        left=0,
        right=1,
        bottom=0,
        top=0.93
    )

    output_file = (
        OUTPUT_DIR
        / f"mapa_exportaciones_{year}.png"
    )

    fig.savefig(
        output_file,
        dpi=100,
        facecolor="white"
    )

    plt.close(fig)

print(
    f"{len(years_export)} mapas guardados en:",
    OUTPUT_DIR
)

## Crear video MP4

In [ ]:
video_file = (
    OUTPUT_DIR
    / "mapa_exportaciones_2017_2025.mp4"
)

with imageio.get_writer(
    video_file,
    fps=24,
    codec="libx264",
    quality=8
) as writer:

    for year in years_export:

        frame = imageio.imread(
            OUTPUT_DIR
            / f"mapa_exportaciones_{year}.png"
        )

        for _ in range(24):
            writer.append_data(frame)

print("Video guardado en:")
print(video_file)

## Importar librerías

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

## Mejorar visualización de los dataframes y los gráficos

In [ ]:
pd.options.display.max_columns = None
pd.options.display.float_format = "{:,.2f}".format

sns.set_style("whitegrid")
%matplotlib inline

## Definir rutas del proyecto

In [ ]:
archivo_entrada = Path(
    "data/processed/final/exports_COL_2017_2026_HS6_country.parquet"
)

PROJECT_ROOT = next(
    (
        ruta
        for ruta in [Path.cwd(), *Path.cwd().parents]
        if (ruta / archivo_entrada).exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Abre VS Code en la carpeta proyecto_comercio_ia."
    )

DATA_FILE = PROJECT_ROOT / archivo_entrada
OUTPUT_DIR = PROJECT_ROOT / "notebook" / "graft"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Proyecto:", PROJECT_ROOT)
print("Entrada:", DATA_FILE)
print("Salidas:", OUTPUT_DIR)

## Cargar el dataset

In [ ]:
df = pd.read_parquet(DATA_FILE)

print("Dimensiones:", df.shape)
df.head()

## Estadísticas descriptivas

### 1. Resumen general

In [ ]:
ultimo_anio = int(
    df["year"].max()
)

ultimo_mes = int(
    df.loc[
        df["year"].eq(ultimo_anio),
        "month"
    ].max()
)

resumen_general = pd.DataFrame({
    "Indicador": [
        "Número de registros",
        "Primer año disponible",
        "Último año disponible",
        f"Último mes disponible de {ultimo_anio}",
        "Número de países destino",
        "Número de productos HS6",
        "Número de capítulos HS2",
        "Valor exportado total (USD)",
        "Valor exportado total (miles de millones USD)",
        "Peso neto total (kg)"
    ],
    "Valor": [
        len(df),
        int(df["year"].min()),
        ultimo_anio,
        ultimo_mes,
        df["partner_name"].nunique(),
        df["hs6"].nunique(),
        df["hs2"].nunique(),
        df["trade_value_usd"].sum(),
        df["trade_value_usd"].sum() / 1e9,
        df["net_weight_kg"].sum()
    ]
})

display(resumen_general)

### 2. Resumen anual

In [ ]:
resumen_anual = (
    df.groupby("year")
    .agg(
        registros=(
            "trade_value_usd",
            "size"
        ),
        valor_exportado_usd=(
            "trade_value_usd",
            "sum"
        ),
        peso_neto_kg=(
            "net_weight_kg",
            "sum"
        ),
        productos_hs6=(
            "hs6",
            "nunique"
        ),
        paises_destino=(
            "partner_name",
            "nunique"
        ),
        meses_disponibles=(
            "month",
            "nunique"
        )
    )
    .reset_index()
)

resumen_anual[
    "valor_miles_millones_usd"
] = (
    resumen_anual[
        "valor_exportado_usd"
    ]
    / 1e9
)

resumen_anual[
    "estado"
] = np.where(
    resumen_anual[
        "meses_disponibles"
    ].eq(12),
    "Completo",
    "Parcial"
)

display(
    resumen_anual.round(2)
)

### 3. Estadísticas descriptivas básicas

In [ ]:
estadisticas = pd.DataFrame({
    "Valor exportado (USD)": [
        df["trade_value_usd"].count(),
        df["trade_value_usd"].mean(),
        df["trade_value_usd"].median(),
        df["trade_value_usd"].min(),
        df["trade_value_usd"].max(),
        df["trade_value_usd"].std(),
        df["trade_value_usd"].quantile(0.25),
        df["trade_value_usd"].quantile(0.75)
    ],
    "Peso neto (kg)": [
        df["net_weight_kg"].count(),
        df["net_weight_kg"].mean(),
        df["net_weight_kg"].median(),
        df["net_weight_kg"].min(),
        df["net_weight_kg"].max(),
        df["net_weight_kg"].std(),
        df["net_weight_kg"].quantile(0.25),
        df["net_weight_kg"].quantile(0.75)
    ]
}, index=[
    "Observaciones",
    "Media",
    "Mediana",
    "Mínimo",
    "Máximo",
    "Desviación estándar",
    "Primer cuartil (Q1)",
    "Tercer cuartil (Q3)"
])

display(
    estadisticas.round(2)
)

In [ ]:
tablas_file = (
    OUTPUT_DIR
    / "estadisticas_descriptivas_exportaciones.xlsx"
)

with pd.ExcelWriter(
    tablas_file,
    engine="openpyxl"
) as writer:

    resumen_general.to_excel(
        writer,
        sheet_name="Resumen general",
        index=False
    )

    resumen_anual.to_excel(
        writer,
        sheet_name="Resumen anual",
        index=False
    )

    estadisticas.to_excel(
        writer,
        sheet_name="Estadisticas"
    )

print("Tablas guardadas en:")
print(tablas_file)

## Principales productos de exportación

In [ ]:
top10_productos = (
    df.groupby(
        ["hs6", "hs6_name"]
    )["trade_value_usd"]
    .sum()
    .nlargest(10)
    .reset_index()
)

top10_productos["producto"] = (
    top10_productos["hs6"].astype(str)
    + " - "
    + top10_productos[
        "hs6_name"
    ].str.slice(0, 35)
    + "..."
)

top10_productos[
    "valor_miles_millones"
] = (
    top10_productos[
        "trade_value_usd"
    ]
    / 1e9
)

top10_plot = (
    top10_productos
    .sort_values(
        "valor_miles_millones"
    )
)

display(
    top10_productos[
        [
            "hs6",
            "hs6_name",
            "trade_value_usd",
            "valor_miles_millones"
        ]
    ].round(2)
)

plt.figure(
    figsize=(11, 7)
)

plt.barh(
    top10_plot["producto"],
    top10_plot[
        "valor_miles_millones"
    ]
)

plt.title(
    "Principales productos HS6 "
    "exportados por Colombia"
)

plt.xlabel(
    "Miles de millones de USD"
)

plt.ylabel(
    "Producto"
)

plt.grid(
    axis="x",
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "04_top10_productos_hs6.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Evolución anual de las exportaciones

In [ ]:
valor_anual = (
    df.groupby("year")[
        "trade_value_usd"
    ]
    .sum()
    .div(1e9)
)

meses_por_anio = (
    df.groupby("year")["month"]
    .nunique()
)

anios_completos = (
    meses_por_anio[
        meses_por_anio.eq(12)
    ]
    .index
)

anios_parciales = (
    meses_por_anio[
        meses_por_anio.lt(12)
    ]
    .index
)

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    anios_completos,
    valor_anual.loc[
        anios_completos
    ],
    marker="o",
    linewidth=2
)

for year in anios_parciales:

    year = int(year)

    if year - 1 in valor_anual.index:

        plt.plot(
            [year - 1, year],
            [
                valor_anual.loc[year - 1],
                valor_anual.loc[year]
            ],
            linestyle="--",
            marker="o",
            fillstyle="none",
            linewidth=1.5
        )

    else:

        plt.scatter(
            year,
            valor_anual.loc[year],
            marker="o",
            facecolors="none"
        )

    ultimo_mes = int(
        df.loc[
            df["year"].eq(year),
            "month"
        ].max()
    )

    meses = {
        1: "enero",
        2: "febrero",
        3: "marzo",
        4: "abril",
        5: "mayo",
        6: "junio",
        7: "julio",
        8: "agosto",
        9: "septiembre",
        10: "octubre",
        11: "noviembre",
        12: "diciembre"
    }

    plt.annotate(
        f"{year} parcial "
        f"(hasta {meses[ultimo_mes]})",
        (
            year,
            valor_anual.loc[year]
        ),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=9
    )

plt.title(
    "Evolución de las "
    "exportaciones de Colombia"
)

plt.xlabel("Año")
plt.ylabel(
    "Miles de millones de USD"
)

plt.xticks(
    valor_anual.index
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "05_evolucion_exportaciones.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Diversificación geográfica por producto

In [ ]:
productos = (
    df.groupby(
        ["hs6", "hs6_name"]
    )
    .agg(
        valor_exportado=(
            "trade_value_usd",
            "sum"
        ),
        peso_neto=(
            "net_weight_kg",
            "sum"
        ),
        paises_destino=(
            "partner_name",
            "nunique"
        )
    )
    .reset_index()
)

productos_plot = (
    productos
    .nlargest(
        150,
        "valor_exportado"
    )
    .copy()
)

productos_plot[
    "valor_bn_usd"
] = (
    productos_plot[
        "valor_exportado"
    ]
    / 1e9
)

productos_plot[
    "bubble_size"
] = (
    np.log1p(
        productos_plot[
            "peso_neto"
        ]
    )
    * 12
)

plt.figure(
    figsize=(12, 7)
)

plt.scatter(
    productos_plot[
        "paises_destino"
    ],
    productos_plot[
        "valor_bn_usd"
    ],
    s=productos_plot[
        "bubble_size"
    ],
    alpha=0.45,
    edgecolors="black",
    linewidth=0.4
)

plt.title(
    "Valor exportado y "
    "diversificación geográfica "
    "por producto HS6",
    fontsize=14
)

plt.xlabel(
    "Número de países destino"
)

plt.ylabel(
    "Valor exportado acumulado "
    "(miles de millones USD)"
)

plt.grid(
    alpha=0.2
)

top10 = productos_plot.nlargest(
    10,
    "valor_exportado"
)

for _, fila in top10.iterrows():

    etiqueta = (
        fila["hs6_name"][:3]
        .upper()
    )

    plt.annotate(
        etiqueta,
        (
            fila["paises_destino"],
            fila["valor_bn_usd"]
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "06_diversificacion_productos.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Importar librerías

In [ ]:
import warnings
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px

import seaborn as sns

sns.set_style("whitegrid")

warnings.filterwarnings("ignore")

## Mejorar visualización

In [ ]:
pd.options.display.max_columns = None
pd.options.display.float_format = "{:,.2f}".format

%matplotlib inline

## Definir rutas del proyecto

In [ ]:
archivo_entrada = Path(
    "data/processed/final/base_model_2017_2024.parquet"
)

PROJECT_ROOT = next(
    (
        ruta
        for ruta in [Path.cwd(), *Path.cwd().parents]
        if (ruta / archivo_entrada).exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Abre VS Code en la carpeta proyecto_comercio_ia."
    )

DATA_FILE = PROJECT_ROOT / archivo_entrada
OUTPUT_DIR = PROJECT_ROOT / "notebook" / "graft"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Proyecto:", PROJECT_ROOT)
print("Entrada:", DATA_FILE)
print("Salidas:", OUTPUT_DIR)

## Conectar DuckDB a la base

In [ ]:
con = duckdb.connect()

ruta_sql = DATA_FILE.as_posix().replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW base AS
    SELECT *
    FROM read_parquet('{ruta_sql}')
""")

estructura = con.sql("""
    SELECT
        COUNT(*) AS filas,
        COUNT(DISTINCT year) AS anios,
        COUNT(DISTINCT iso3_d) AS paises,
        COUNT(DISTINCT hs6) AS productos_hs6
    FROM base
""").df()

estructura

## 1. Evolución del margen extensivo

La variable `exported` permite observar qué proporción de las combinaciones **producto–país** registra exportaciones positivas en cada año.


In [ ]:
margen_anual = con.sql("""
    SELECT
        year,
        COUNT(*) AS combinaciones,
        SUM(exported) AS combinaciones_exportadas,
        AVG(exported) * 100 AS pct_exportadas,
        SUM(export_value_usd) / 1e9 AS exportaciones_bn_usd,
        SUM(import_demand_usd) / 1e9 AS demanda_bn_usd
    FROM base
    GROUP BY year
    ORDER BY year
""").df()

display(margen_anual.round(2))

plt.figure(figsize=(10, 5))

plt.plot(
    margen_anual["year"],
    margen_anual["pct_exportadas"],
    marker="o",
    linewidth=2
)

plt.title(
    "Participación de combinaciones producto–país con exportaciones"
)

plt.xlabel("Año")
plt.ylabel("Combinaciones exportadas (%)")
plt.xticks(margen_anual["year"])
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "08_01_margen_extensivo.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 2. Intensidad exportadora por destino y año

Se seleccionan los principales destinos según el valor exportado acumulado y se compara su evolución entre 2017 y 2024.


In [ ]:
top_paises = con.sql("""
    SELECT
        iso3_d,
        SUM(export_value_usd) AS exportaciones
    FROM base
    GROUP BY iso3_d
    ORDER BY exportaciones DESC
    LIMIT 20
""").df()

lista_paises = ", ".join(
    f"'{x}'"
    for x in top_paises["iso3_d"]
)

destino_anual = con.sql(f"""
    SELECT
        year,
        iso3_d,
        SUM(export_value_usd) / 1e9 AS exportaciones_bn_usd
    FROM base
    WHERE iso3_d IN ({lista_paises})
    GROUP BY year, iso3_d
    ORDER BY year, iso3_d
""").df()

heat = (
    destino_anual
    .pivot(
        index="iso3_d",
        columns="year",
        values="exportaciones_bn_usd"
    )
    .fillna(0)
)

heat = heat.loc[
    top_paises["iso3_d"]
]

plt.figure(figsize=(10, 8))

img = plt.imshow(
    heat.values,
    aspect="auto"
)

plt.colorbar(
    img,
    label="Miles de millones de USD"
)

plt.xticks(
    range(len(heat.columns)),
    heat.columns
)

plt.yticks(
    range(len(heat.index)),
    heat.index
)

plt.title(
    "Intensidad exportadora hacia los 20 principales destinos"
)

plt.xlabel("Año")
plt.ylabel("País destino")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "08_02_heatmap_destinos.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 3. Diversificación geográfica por producto

Se compara el valor exportado acumulado de cada producto con el número de mercados en los que efectivamente fue exportado.


In [ ]:
productos = con.sql("""
    SELECT
        hs6,
        ANY_VALUE(hs6_name) AS hs6_name,
        SUM(export_value_usd) AS export_value_usd,
        SUM(import_demand_usd) AS import_demand_usd,
        COUNT(
            DISTINCT CASE
                WHEN exported = 1
                THEN iso3_d
            END
        ) AS mercados_exportados
    FROM base
    GROUP BY hs6
""").df()

productos_plot = (
    productos
    .nlargest(
        250,
        "export_value_usd"
    )
    .copy()
)

productos_plot["export_bn"] = (
    productos_plot["export_value_usd"]
    / 1e9
)

demanda_log = np.log1p(
    productos_plot["import_demand_usd"]
)

productos_plot["bubble_size"] = (
    20
    + 180
    * (
        demanda_log
        - demanda_log.min()
    )
    / (
        demanda_log.max()
        - demanda_log.min()
    )
)

plt.figure(figsize=(11, 7))

plt.scatter(
    productos_plot["mercados_exportados"],
    productos_plot["export_bn"],
    s=productos_plot["bubble_size"],
    alpha=0.45,
    edgecolors="black",
    linewidth=0.4
)

plt.title(
    "Diversificación geográfica de los principales productos HS6"
)

plt.xlabel("Número de mercados exportados")
plt.ylabel("Exportaciones acumuladas (miles de millones USD)")
plt.grid(alpha=0.2)

top_labels = productos_plot.nlargest(
    12,
    "export_value_usd"
)

for _, fila in top_labels.iterrows():

    etiqueta = str(
        fila["hs6_name"]
    )[:3].upper()

    plt.annotate(
        etiqueta,
        (
            fila["mercados_exportados"],
            fila["export_bn"]
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "08_03_diversificacion_productos.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 4. Demanda externa no atendida

Esta visualización usa únicamente combinaciones con `exported = 0`.  
No representa una predicción del modelo: muestra **demanda importadora observada en mercados donde Colombia no exportó ese producto**.


In [ ]:
oportunidad_producto = con.sql("""
    SELECT
        hs6,
        ANY_VALUE(hs6_name) AS hs6_name,
        SUM(import_demand_usd) AS demanda_no_atendida,
        COUNT(DISTINCT iso3_d) AS mercados_potenciales
    FROM base
    WHERE exported = 0
      AND import_demand_usd > 0
    GROUP BY hs6
    ORDER BY demanda_no_atendida DESC
    LIMIT 20
""").df()

oportunidad_producto["demanda_bn"] = (
    oportunidad_producto["demanda_no_atendida"]
    / 1e9
)

oportunidad_producto["etiqueta"] = (
    oportunidad_producto["hs6"].astype(str)
    + " - "
    + oportunidad_producto["hs6_name"]
      .str.slice(0, 42)
)

plot_oportunidad = (
    oportunidad_producto
    .sort_values("demanda_bn")
)

display(
    oportunidad_producto[
        [
            "hs6",
            "hs6_name",
            "demanda_bn",
            "mercados_potenciales"
        ]
    ].round(2)
)

plt.figure(figsize=(11, 8))

plt.barh(
    plot_oportunidad["etiqueta"],
    plot_oportunidad["demanda_bn"]
)

plt.title(
    "Productos con mayor demanda importadora en mercados no atendidos"
)

plt.xlabel(
    "Demanda acumulada no atendida (miles de millones USD)"
)

plt.ylabel("Producto HS6")
plt.grid(axis="x", alpha=0.2)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "08_04_demanda_no_atendida.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 5. Distancia y probabilidad observada de exportar

Para evitar que los millones de combinaciones dominen el gráfico, primero se calcula una tasa de exportación por país y luego se agrupan los destinos en deciles de distancia.


In [ ]:
gravedad_pais = con.sql("""
    SELECT
        iso3_d,
        AVG(dist) AS dist,
        AVG(exported) * 100 AS pct_exportado
    FROM base
    WHERE dist IS NOT NULL
    GROUP BY iso3_d
""").df()

gravedad_pais["decil_distancia"] = pd.qcut(
    gravedad_pais["dist"],
    q=10,
    duplicates="drop"
)

gravedad = (
    gravedad_pais
    .groupby(
        "decil_distancia",
        observed=True
    )
    .agg(
        distancia_media=("dist", "mean"),
        pct_exportado=("pct_exportado", "mean"),
        paises=("iso3_d", "nunique")
    )
    .reset_index()
)

plt.figure(figsize=(10, 5))

plt.plot(
    gravedad["distancia_media"],
    gravedad["pct_exportado"],
    marker="o",
    linewidth=2
)

plt.title(
    "Distancia y participación observada de combinaciones exportadas"
)

plt.xlabel("Distancia media")
plt.ylabel("Combinaciones exportadas (%)")
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "08_05_distancia_exportacion.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 6. Idioma común y acuerdo comercial

Comparación descriptiva de la proporción de combinaciones exportadas según idioma oficial común (`comlang_off`) y acuerdo comercial (`fta_wto`).


In [ ]:
idioma_fta = con.sql("""
    SELECT
        CAST(comlang_off AS INTEGER) AS idioma_comun,
        CAST(fta_wto AS INTEGER) AS acuerdo_comercial,
        AVG(exported) * 100 AS pct_exportado,
        COUNT(*) AS observaciones
    FROM base
    WHERE comlang_off IS NOT NULL
      AND fta_wto IS NOT NULL
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

idioma_fta["grupo"] = (
    "Idioma="
    + idioma_fta["idioma_comun"].astype(str)
    + " | FTA="
    + idioma_fta["acuerdo_comercial"].astype(str)
)

display(idioma_fta.round(2))

plt.figure(figsize=(9, 5))

plt.bar(
    idioma_fta["grupo"],
    idioma_fta["pct_exportado"]
)

plt.title(
    "Exportación observada según idioma común y acuerdo comercial"
)

plt.xlabel("Combinación")
plt.ylabel("Combinaciones exportadas (%)")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "08_06_idioma_fta.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 7. Tamaño de mercado y exportaciones

Se usa el último año de la base para comparar tamaño económico del destino, demanda importadora y exportaciones colombianas.


In [ ]:
ultimo_anio = int(
    con.sql(
        "SELECT MAX(year) FROM base"
    ).fetchone()[0]
)

mercados = con.sql(f"""
    SELECT
        iso3_d,
        SUM(export_value_usd) AS export_value_usd,
        SUM(import_demand_usd) AS import_demand_usd,
        MAX(gdp_d) AS gdp_d,
        MAX(pop_d) AS pop_d
    FROM base
    WHERE year = {ultimo_anio}
      AND gdp_d > 0
      AND import_demand_usd > 0
    GROUP BY iso3_d
""").df()

mercados = mercados[
    mercados["gdp_d"].gt(0)
    & mercados["import_demand_usd"].gt(0)
].copy()

mercados["export_bn"] = (
    mercados["export_value_usd"]
    / 1e9
)

mercados["demand_bn"] = (
    mercados["import_demand_usd"]
    / 1e9
)

plt.figure(figsize=(10, 7))

plt.scatter(
    mercados["demand_bn"],
    mercados["export_bn"],
    alpha=0.55,
    s=45
)

plt.xscale("log")
plt.yscale("symlog", linthresh=0.001)

plt.title(
    f"Demanda del mercado y exportaciones colombianas — {ultimo_anio}"
)

plt.xlabel(
    "Demanda importadora del destino (miles de millones USD, escala log)"
)

plt.ylabel(
    "Exportaciones de Colombia (miles de millones USD)"
)

plt.grid(alpha=0.2)

top_mercados = mercados.nlargest(
    12,
    "export_value_usd"
)

for _, fila in top_mercados.iterrows():

    plt.annotate(
        fila["iso3_d"],
        (
            fila["demand_bn"],
            fila["export_bn"]
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "08_07_tamano_mercado.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 8. Relaciones multivariadas

Se calcula una matriz de correlación de Spearman sobre una muestra de 250.000 observaciones.  
Se utilizan transformaciones logarítmicas para las variables monetarias y de tamaño.


In [ ]:
corr_sample = con.sql("""
    SELECT
        import_demand_usd,
        export_value_usd,
        dist,
        gdp_d,
        gdpcap_d,
        pop_d,
        scaled_sci_2021,
        comrelig,
        partner,
        X_NoComb,
        M_NoComb,
        fta_wto,
        comlang_off,
        exported
    FROM base
    USING SAMPLE 250000 ROWS
""").df()

corr_data = pd.DataFrame({
    "ln_demanda": np.log1p(
        corr_sample["import_demand_usd"]
    ),
    "ln_exportaciones": np.log1p(
        corr_sample["export_value_usd"]
    ),
    "ln_distancia": np.log1p(
        corr_sample["dist"]
    ),
    "ln_gdp_destino": np.log1p(
        corr_sample["gdp_d"]
    ),
    "ln_gdpcap_destino": np.log1p(
        corr_sample["gdpcap_d"]
    ),
    "ln_poblacion_destino": np.log1p(
        corr_sample["pop_d"]
    ),
    "conexion_cientifica": corr_sample[
        "scaled_sci_2021"
    ],
    "religion_comun": corr_sample[
        "comrelig"
    ],
    "partner": corr_sample[
        "partner"
    ],
    "X_NoComb": np.log1p(
        corr_sample["X_NoComb"]
    ),
    "M_NoComb": np.log1p(
        corr_sample["M_NoComb"]
    ),
    "FTA": corr_sample[
        "fta_wto"
    ],
    "idioma_comun": corr_sample[
        "comlang_off"
    ],
    "exported": corr_sample[
        "exported"
    ]
})

corr = corr_data.corr(
    method="spearman"
)

plt.figure(figsize=(11, 9))

img = plt.imshow(
    corr.values,
    vmin=-1,
    vmax=1
)

plt.colorbar(
    img,
    label="Correlación de Spearman"
)

plt.xticks(
    range(len(corr.columns)),
    corr.columns,
    rotation=75,
    ha="right"
)

plt.yticks(
    range(len(corr.index)),
    corr.index
)

plt.title(
    "Relaciones entre variables seleccionadas del modelo"
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "08_08_correlaciones.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 9. Explorador interactivo de oportunidades

Se muestran las combinaciones **producto–país no exportadas** con mayor demanda importadora en el último año.  
El gráfico se guarda como HTML para poder abrirlo en el navegador y explorar cada observación.


In [ ]:
oportunidades = con.sql(f"""
    SELECT
        hs6,
        hs6_name,
        iso3_d,
        import_demand_usd,
        dist,
        gdp_d,
        scaled_sci_2021,
        comlang_off,
        fta_wto
    FROM base
    WHERE year = {ultimo_anio}
      AND exported = 0
      AND import_demand_usd > 0
      AND gdp_d > 0
    ORDER BY import_demand_usd DESC
    LIMIT 5000
""").df()

oportunidades["demanda_millones"] = (
    oportunidades["import_demand_usd"]
    / 1e6
)

oportunidades["gdp_bn"] = (
    oportunidades["gdp_d"]
    / 1e9
)

fig = px.scatter(
    oportunidades,
    x="demanda_millones",
    y="gdp_bn",
    hover_name="hs6_name",
    hover_data=[
        "hs6",
        "iso3_d",
        "dist",
        "comlang_off",
        "fta_wto"
    ],
    log_x=True,
    log_y=True,
    title=(
        "Oportunidades no exportadas: "
        f"demanda y tamaño de mercado — {ultimo_anio}"
    ),
    labels={
        "demanda_millones":
            "Demanda importadora (millones USD)",
        "gdp_bn":
            "PIB del destino (miles de millones USD)"
    }
)

fig.show()

html_file = (
    OUTPUT_DIR
    / "08_09_oportunidades_interactivas.html"
)

fig.write_html(
    html_file,
    include_plotlyjs="cdn"
)

print("Archivo interactivo guardado en:")
print(html_file)

In [ ]:
distancia = con.sql("""
    SELECT
        iso3_d,
        AVG(dist) AS dist,
        AVG(exported) * 100 AS pct_exportado
    FROM base
    WHERE dist IS NOT NULL
    GROUP BY iso3_d
""").df()

distancia["grupo_distancia"] = pd.qcut(
    distancia["dist"],
    q=5,
    labels=[
        "Muy cerca",
        "Cerca",
        "Media",
        "Lejos",
        "Muy lejos"
    ]
)

plt.figure(figsize=(9, 5))

sns.barplot(
    data=distancia,
    x="grupo_distancia",
    y="pct_exportado"
)

plt.title(
    "Probabilidad observada de exportar según distancia"
)

plt.xlabel("Distancia relativa del mercado")
plt.ylabel("Combinaciones exportadas (%)")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "08_10_exportacion_distancia_grupos.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
demanda_sample = con.sql("""
    SELECT
        exported,
        import_demand_usd
    FROM base
    WHERE import_demand_usd > 0
    USING SAMPLE 200000 ROWS
""").df()

demanda_sample["ln_demanda"] = np.log1p(
    demanda_sample["import_demand_usd"]
)

demanda_sample["estado"] = demanda_sample[
    "exported"
].map({
    0: "No exporta",
    1: "Exporta"
})

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=demanda_sample,
    x="estado",
    y="ln_demanda",
    showfliers=False
)

plt.title(
    "Demanda importadora según presencia exportadora de Colombia"
)

plt.xlabel("")
plt.ylabel("Log de demanda importadora")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "08_11_demanda_exporta_boxplot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
matriz_oportunidad = con.sql(f"""
    SELECT
        hs6,
        ANY_VALUE(hs6_name) AS hs6_name,
        COUNT(DISTINCT iso3_d) AS mercados_no_atendidos,
        AVG(import_demand_usd) AS demanda_media,
        SUM(import_demand_usd) AS demanda_total
    FROM base
    WHERE year = {ultimo_anio}
      AND exported = 0
      AND import_demand_usd > 0
    GROUP BY hs6
""").df()

matriz_oportunidad = (
    matriz_oportunidad
    .nlargest(
        500,
        "demanda_total"
    )
    .copy()
)

matriz_oportunidad["demanda_media_m"] = (
    matriz_oportunidad["demanda_media"]
    / 1e6
)

matriz_oportunidad["demanda_total_bn"] = (
    matriz_oportunidad["demanda_total"]
    / 1e9
)

x_med = matriz_oportunidad[
    "mercados_no_atendidos"
].median()

y_med = matriz_oportunidad[
    "demanda_media_m"
].median()

plt.figure(figsize=(11, 7))

plt.scatter(
    matriz_oportunidad["mercados_no_atendidos"],
    matriz_oportunidad["demanda_media_m"],
    s=(
        30
        + matriz_oportunidad["demanda_total_bn"] * 10
    ),
    alpha=0.45,
    edgecolors="black",
    linewidth=0.4
)

plt.axvline(
    x=x_med,
    linestyle="--",
    linewidth=1
)

plt.axhline(
    y=y_med,
    linestyle="--",
    linewidth=1
)

plt.yscale("log")

plt.title(
    f"Matriz de oportunidades de exportación — {ultimo_anio}"
)

plt.xlabel(
    "Número de mercados con demanda no atendida"
)

plt.ylabel(
    "Demanda media por mercado (millones USD)"
)

top_labels = matriz_oportunidad.nlargest(
    12,
    "demanda_total"
)

for _, fila in top_labels.iterrows():

    plt.annotate(
        str(fila["hs6_name"])[:3].upper(),
        (
            fila["mercados_no_atendidos"],
            fila["demanda_media_m"]
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "08_12_matriz_oportunidades.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
dist_fta = con.sql("""
    SELECT
        iso3_d,
        AVG(dist) AS dist,
        MAX(fta_wto) AS fta_wto,
        AVG(exported) * 100 AS pct_exportado
    FROM base
    WHERE dist IS NOT NULL
      AND fta_wto IS NOT NULL
    GROUP BY iso3_d
""").df()

dist_fta["distancia"] = pd.qcut(
    dist_fta["dist"],
    q=4,
    labels=[
        "Corta",
        "Media-baja",
        "Media-alta",
        "Larga"
    ]
)

dist_fta["acuerdo"] = np.where(
    dist_fta["fta_wto"] == 1,
    "Con acuerdo",
    "Sin acuerdo"
)

heat_fta = (
    dist_fta
    .pivot_table(
        index="distancia",
        columns="acuerdo",
        values="pct_exportado",
        observed=True
    )
)

plt.figure(figsize=(7, 5))

sns.heatmap(
    heat_fta,
    annot=True,
    fmt=".1f",
    cmap="Reds",
    cbar_kws={
        "label": "Combinaciones exportadas (%)"
    }
)

plt.title(
    "Presencia exportadora según distancia y acuerdo comercial"
)

plt.xlabel("")
plt.ylabel("Distancia")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "08_13_heatmap_distancia_fta.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
ranking = con.sql("""
    WITH anual AS (
        SELECT
            year,
            iso3_d,
            SUM(export_value_usd) AS exportaciones
        FROM base
        GROUP BY year, iso3_d
    ),

    ranking AS (
        SELECT
            *,
            RANK() OVER (
                PARTITION BY year
                ORDER BY exportaciones DESC
            ) AS rank
        FROM anual
    )

    SELECT *
    FROM ranking
    WHERE rank <= 10
    ORDER BY year, rank
""").df()

paises_ranking = (
    ranking.groupby("iso3_d")["exportaciones"]
    .sum()
    .nlargest(12)
    .index
)

ranking_plot = ranking[
    ranking["iso3_d"].isin(paises_ranking)
].copy()

plt.figure(figsize=(12, 7))

for pais, grupo in ranking_plot.groupby("iso3_d"):

    grupo = grupo.sort_values("year")

    plt.plot(
        grupo["year"],
        grupo["rank"],
        marker="o",
        linewidth=2,
        label=pais
    )

plt.gca().invert_yaxis()

plt.yticks(
    range(1, 11)
)

plt.xticks(
    sorted(ranking_plot["year"].unique())
)

plt.title(
    "Evolución del ranking de los principales destinos"
)

plt.xlabel("Año")
plt.ylabel("Posición")

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.grid(alpha=0.2)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "08_14_ranking_destinos.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
mercados_interactivos = con.sql(f"""
    SELECT
        iso3_d,
        SUM(import_demand_usd) / 1e9 AS demanda_bn,
        SUM(export_value_usd) / 1e9 AS exportaciones_bn,
        AVG(dist) AS distancia,
        AVG(exported) * 100 AS pct_productos_exportados,
        MAX(fta_wto) AS fta_wto,
        MAX(comlang_off) AS comlang_off
    FROM base
    WHERE year = {ultimo_anio}
    GROUP BY iso3_d
""").df()

mercados_interactivos["FTA"] = np.where(
    mercados_interactivos["fta_wto"] == 1,
    "Con acuerdo",
    "Sin acuerdo"
)

fig = px.scatter(
    mercados_interactivos,
    x="distancia",
    y="demanda_bn",
    size="exportaciones_bn",
    color="FTA",
    hover_name="iso3_d",
    hover_data={
        "pct_productos_exportados": ":.2f",
        "demanda_bn": ":.2f",
        "exportaciones_bn": ":.2f",
        "distancia": ":.0f",
        "comlang_off": True
    },
    log_y=True,
    size_max=55,
    title=(
        f"Mercados internacionales: "
        f"demanda, distancia y presencia colombiana — {ultimo_anio}"
    ),
    labels={
        "distancia": "Distancia",
        "demanda_bn": "Demanda importadora (miles de millones USD)",
        "exportaciones_bn": "Exportaciones de Colombia",
        "pct_productos_exportados":
            "% de combinaciones exportadas"
    }
)

fig.show()

fig.write_html(
    OUTPUT_DIR / "08_15_mercados_interactivos.html",
    include_plotlyjs="cdn"
)